In [2]:
import kagglehub
import os
import pandas as pd
# from datasets import load_dataset
# from kagglehub import KaggleDatasetAdapter

In [3]:
def download_dataset()->list[str]:
    """
    Download the dataset from Kaggle and return the paths to the files.
    """
    dataset_dir = kagglehub.dataset_download("josephleake/huge-collection-of-reddit-votes")
    paths = []
    for dir_path, _, file_names in os.walk(dataset_dir):
        for file_name in file_names:
            paths.append(os.path.join(dir_path, file_name))
    print(f'File path to votes:\n{paths[0]}')
    print(f'File path to submissions:\n{paths[1]}')
    return paths

def get_dataframe()->tuple[pd.DataFrame]:
    """
    Return a tuple of two pandas.Dataframe: votes and submissions.

    Returns:
        tuple[pd.DataFrame]: a tuple of two dataframes.
    """
    paths = download_dataset()
    votes = pd.read_csv(paths[0], sep='\t')
    submissions = pd.read_csv(paths[1], sep='\t')
    return (votes, submissions)

def view_users_votes(votes:pd.DataFrame):
    view = (
        votes
        .groupby(['USERNAME', 'SUBREDDIT', 'VOTE'])
        .size()                         # count upvotes/downvotes in each group
        .unstack(fill_value=0)          # pivot VOTE labels into columns
        .rename(columns={
            'upvote':   'num_upvotes',
            'downvote': 'num_downvotes'
        })
        .reset_index()                  # turn USERNAME & SUBREDDIT back into columns
    )
    return view

def view_subreddit_counts(submissions:pd.DataFrame):
    view = (
        submissions
        .groupby(['SUBREDDIT'])
        .size()                         # count upvotes/downvotes in each group
        # .unstack(fill_value=0)          # pivot VOTE labels into columns
                        # turn USERNAME & SUBREDDIT back into columns
    )
    return view

In [4]:
votes, submissions = get_dataframe()

100%|██████████| 2.28G/2.28G [03:34<00:00, 11.4MB/s] 

Extracting files...


File path to votes:
C:\Users\z5612172.ADUNSW\.cache\kagglehub\datasets\josephleake\huge-collection-of-reddit-votes\versions\1\44_million_reddit_votes\44_million_votes.txt
File path to submissions:
C:\Users\z5612172.ADUNSW\.cache\kagglehub\datasets\josephleake\huge-collection-of-reddit-votes\versions\1\submission_info\submission_info.txt


In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv('reddit_showerthoughts.tsv', sep='\t', on_bad_lines='skip')
popular_posts = (
    df.sort_values(by="ups", ascending=False)
           .head(200)["submission_id"]
           .tolist()
)

def get_upvoted_posts_by_user(votes, test_df, subreddit="r/Showerthoughts"):
        filtered = votes[
            (votes["SUBREDDIT"] == subreddit) &
            (votes["VOTE"] == "upvote") &
            (votes["SUBMISSION_ID"].isin(set(df["submission_id"])))
        ]
        return filtered.groupby("USERNAME")["SUBMISSION_ID"].apply(set).to_dict()

actual_votes_positive = get_upvoted_posts_by_user(votes, df)

# Evaluate popularity baseline
total_ndcg, recommended_items = 0, set()
all_items = set(df['submission_id'])
user_count = 0

def ndcg_at_k(recommended, actual_upvotes, k):
    if not actual_upvotes:
        return 0.0
    # create relevance scores 
    relevance = [1 if item in actual_upvotes else 0 for item in recommended[:k]]
    # dcg
    dcg = sum((2 ** rel - 1) / np.log2(idx + 2) 
             for idx, rel in enumerate(relevance))
    #  idcg
    ideal_relevance = sorted([1] * min(len(actual_upvotes), k), reverse=True)
    idcg = sum((2 ** rel - 1) / np.log2(idx + 2) 
              for idx, rel in enumerate(ideal_relevance))

    return dcg / idcg if idcg > 0 else 0
testing_users = ['Raven2002', 'mguardian_north', 'Clen23', 'Reeses2150', 'spockspeare', 'apoeticturtle', 'Mash404', 'locks_are_paranoid', 
                 'daygloviking', 'thx1138jr', 'Livelogikal', 'MingeyMackrel', 'uncertainusurper', 
                 'lokier01', 'baddonkey', 'pierrekrahn', 'CubyChris', 'Adventurous_Guy', 'stratman42', 'VerbotenPublish']

for user in actual_votes_positive:
    if user not in testing_users:
        continue
    recs = popular_posts
    actual_positive = actual_votes_positive.get(user, set())
    ndcg = ndcg_at_k(recs, actual_positive, 200)
    print("ndcg for user", user, ndcg)
    total_ndcg += ndcg
    recommended_items.update(recs)
    user_count += 1



print("Popularity Baseline Results:")
print(f"NDCG@200: {total_ndcg / user_count if user_count else 0:.4f}")
print(f"Users evaluated: {user_count}")

ndcg for user Adventurous_Guy 0.035397102679812505
ndcg for user Clen23 0.028205674704978298
ndcg for user CubyChris 0.0874781957916163
ndcg for user Livelogikal 0.04754629780238845
ndcg for user Mash404 0.013487334602603039
ndcg for user MingeyMackrel 0.0
ndcg for user Raven2002 0.0
ndcg for user Reeses2150 0.11317338678226731
ndcg for user VerbotenPublish 0.0
ndcg for user apoeticturtle 0.006630439342278273
ndcg for user baddonkey 0.0
ndcg for user daygloviking 0.0
ndcg for user locks_are_paranoid 0.03375419620283036
ndcg for user lokier01 0.18062146562720927
ndcg for user mguardian_north 0.012953243923398758
ndcg for user pierrekrahn 0.0833826679146569
ndcg for user spockspeare 0.026017818108579855
ndcg for user stratman42 0.012628581457119266
ndcg for user thx1138jr 0.0
ndcg for user uncertainusurper 0.031714513314744626
Popularity Baseline Results:
NDCG@200: 0.0356
Users evaluated: 20


In [4]:
votes[400:500]

,SUBMISSION_ID,SUBREDDIT,CREATED_TIME,USERNAME,VOTE
400,t3_dyudhy,r/13or30,NaN,-Rick_Sanchez_,upvote
401,t3_dzmkff,r/KidsAreFuckingStupid,NaN,-Rick_Sanchez_,upvote
402,t3_dz14rj,r/maybemaybemaybe,NaN,-Rick_Sanchez_,upvote
403,t3_dynyrs,r/classicwow,NaN,-Rick_Sanchez_,upvote
404,t3_dyl1v4,r/MyPeopleNeedMe,NaN,-Rick_Sanchez_,upvote
...,...,...,...,...,...
495,t3_dyooji,r/gifs,NaN,-Rick_Sanchez_,upvote
496,t3_dy0o9m,r/aww,NaN,-Rick_Sanchez_,upvote
497,t3_dym6dw,r/dogswithjobs,NaN,-Rick_Sanchez_,upvote
498,t3_dz43xt,r/therewasanattempt,NaN,-Rick_Sanchez_,upvote


In [5]:
user_votes_view = view_users_votes(votes)

In [ ]:
user_votes_view.head(100)

VOTE,USERNAME,SUBREDDIT,num_downvotes,num_upvotes
0,---UUU---,r/Artistic_Hentai,5,2
1,---UUU---,r/AsianNSFW,1,0
2,---UUU---,r/BikiniBottomTwitter,0,1
3,---UUU---,r/CelebEconomy,0,10
4,---UUU---,r/ClashOfClans,16,3
...,...,...,...,...
95,--NiNjA--,r/GalaxyS6,1,0
96,--NiNjA--,r/GalaxyWatch,1,13
97,--NiNjA--,r/Gamingcirclejerk,2,0
98,--NiNjA--,r/GetMotivated,4,7


In [20]:
submission_view=view_subreddit_counts(submissions).sort_values(ascending=False)
submission_view.head(15)

SUBREDDIT
funny              354280
AskReddit          257056
politics           229341
memes              214247
pics               178552
The_Donald         169427
dankmemes          164220
gaming             143963
aww                142530
leagueoflegends    111667
gonewild           104711
videos             102076
AdviceAnimals       98440
Showerthoughts      95153
nba                 85376
dtype: int64

In [ ]:
def filter_subreddits(
    votes: pd.DataFrame,
    num_upvotes: int = 0,
    num_downvotes: int = 0,
    total_votes: int = 0,
    num_users: int = 0,
) -> pd.DataFrame:
    """
    Filter a DataFrame of subreddit vote counts according to given thresholds.

    Parameters:
    - votes: DataFrame with at least ['USERNAME', 'SUBREDDIT', 'num_upvotes', 'num_downvotes'] columns.
    - num_upvotes: keep rows where num_upvotes > this value (if > 0).
    - num_downvotes: keep rows where num_downvotes > this value (if > 0).
    - total_votes: keep rows where (num_upvotes + num_downvotes) > this value (if > 0).
    - num_users: keep rows where the subreddit has more than this many unique users (if > 0).

    Returns:
    - Filtered DataFrame.
    """
    # Start with an all-True mask
    mask = pd.Series(True, index=votes.index)

    # Apply upvotes threshold
    if num_upvotes > 0:
        mask &= votes['num_upvotes'] > num_upvotes

    # Apply downvotes threshold
    if num_downvotes > 0:
        mask &= votes['num_downvotes'] > num_downvotes

    # Apply total votes threshold
    if total_votes > 0:
        mask &= (votes['num_upvotes'] + votes['num_downvotes']) > total_votes

    # Apply distinct user count per subreddit threshold
    if num_users > 0:
        # Compute number of unique users for each subreddit
        user_counts = votes.groupby('SUBREDDIT')['USERNAME'].transform('nunique')
        mask &= user_counts > num_users

    # Return a copy of the filtered DataFrame
    return votes[mask].copy()

In [56]:
subreddits = filter_subreddits(user_votes_view, 0, 0, 100, 200)
subreddits['SUBREDDIT'].unique().shape

(2366,)